This notebook builds a unified metadata configuration by combining Oracle information schema, key constraints, and primary key data, and generates structured source queries for downstream in Fabric.

In [ ]:
SchemaName = ''

In [ ]:
################################################### IMPORT  REQUIRED PACKAGES ###################################################
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
col, lit, concat, when, trim, collect_list, row_number, to_date,date_format,
col, concat, lit, to_timestamp, date_format, to_date, coalesce,concat_ws, md5,struct,
udf , concat, to_timestamp,count, when ,  sum as _sum)
from pyspark.sql.window import Window
from datetime import datetime, timezone, timedelta
from delta.tables import DeltaTable
import pandas as pd
from com.microsoft.spark.fabric import Constants
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants
from pyspark.sql.types import (
    DoubleType, IntegerType, BooleanType, StringType,StructField,ArrayType,
    BinaryType, TimestampType, DateType, LongType, StructType
)
import concurrent.futures
import pytz
import traceback
import os
import json
 
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, concat, collect_list, row_number
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)
from pyspark.sql.window import Window

In [ ]:
################################################### SPARK SESSION ###################################################
spark = SparkSession.builder \
    .appName("Oracle") \
    .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .config("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED") \
    .config("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED") \
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED") \
    .config("spark.sql.legacy.parquet.datetimeRebaseModeInWrite", "CORRECTED") \
    .config("spark.sql.legacy.parquet.datetimeRebaseModeInRead", "CORRECTED") \
    .config("spark.sql.caseSensitive", "true") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()
 

In [ ]:
################################################### LOADING METADATA ####################################################
try:
    InformationSchema="SELECT * FROM {ConfigSchemaName}.SourceInformationSchema"
    InformationSchema_df= spark.read.option(Constants.DatabaseName, "WH_MetaData").synapsesql(InformationSchema)
    InformationSchema_df=InformationSchema_df.filter(col("OWNER")=='HOSPITAL')
except Exception as e:
    print(e)
 

In [ ]:
################################################### LOADING KEY CONSTRAINTS ################################################### 
try:
    KeyConstraint = """
        SELECT * 
        FROM {ConfigSchemaName}.KeyConstraintConfig
    """

    KeyConstraint_df = spark.read.option(
        Constants.DatabaseName,
        "WH_MetaData"
    ).synapsesql(KeyConstraint)

    KeyConstraint_df = KeyConstraint_df.filter(
        col("OWNER") == 'HOSPITAL'
    )

except Exception as e:
    print("Error loading Key Constraints:", e)

In [ ]:
################################################### LOADING PRIMARY KEY  ################################################### 

try:
    PrimaryKey = """
        SELECT * 
        FROM {ConfigSchemaName}.PrimayKeyConfig
    """

    PrimaryKey_df = spark.read.option(
        Constants.DatabaseName,
        "WH_MetaData"
    ).synapsesql(PrimaryKey)

    #PrimaryKey_df = PrimaryKey_df.filter(
       # col("OWNER") == 'HOSPITAL'
    #)

except Exception as e:
    print("Error loading Key Constraints:", e)

In [ ]:
 ################################################### 4.FILTER AND GROUP  ###################################################

information_req = InformationSchema_df
#.filter(col("OWNER") == "HOSPITAL"
#)

grouped_df = InformationSchema_df.groupBy("OWNER", "TABLE_NAME").agg(
    collect_list(struct("COLUMN_NAME", "DATA_TYPE")).alias("columns")
)
 

In [ ]:
pk_df = PrimaryKey_df.select(
    "OWNER",
    "TABLE_NAME",
    col("COLUMN_NAME").alias("PrimaryKey")
)

final_with_pk = grouped_df.join(
    pk_df,
    ["OWNER", "TABLE_NAME"],
    "left"
)

In [ ]:
################################################### 5.GENERATING SOURCE QUERY ###################################################
def generate_source_query(table_name, columns, schema_name):
    select_statements = []

    for column in columns:
        column_name = column["COLUMN_NAME"]
        data_type = column["DATA_TYPE"]

        if data_type == "NUMBER":
            select_statements.append(f"CAST({column_name} AS BINARY_DOUBLE) AS {column_name}")
        elif data_type == "DATE":
            select_statements.append(f"CAST({column_name} AS TIMESTAMP) AS {column_name}")
        else:
            select_statements.append(column_name)

    return f"SELECT {', '.join(select_statements)} FROM {schema_name}.{table_name}"

In [ ]:
################################################### 6.COLLECTING METADATA ###################################################
metadata = []
for row in final_with_pk.collect():
    table_name = row["TABLE_NAME"]
    schema_name = row["OWNER"]
    columns = row["columns"]
    source_query = generate_source_query(table_name, columns, schema_name)
    primary_key = row["PrimaryKey"] 
    metadata.append((table_name, schema_name, source_query, primary_key))

In [ ]:
schema = StructType([
    StructField("SourceTableName", StringType(), True),
    StructField("SourceSchemaName", StringType(), True),
    StructField("SourceQuery", StringType(), True),
    StructField("PrimaryKey", StringType(), True)
]) 

In [ ]:
final_df = spark.createDataFrame(metadata, schema=schema)
#Configdf="SELECT Id  FROM Oracle.IncrementalConfigETL "
#max_id= spark.read.option(Constants.DatabaseName, "WH_MetaData").synapsesql(Configdf)
#max_id = max_id.agg({"Id": "max"}).collect()[0]["max(Id)"]
max_id=0

In [ ]:

################################################## 8.COLUMN SELECTION AND MODIFICATIONS ###################################################
 
window_spec = Window.orderBy(lit(1))
 
from pyspark.sql.functions import lit, concat, row_number, col
from pyspark.sql.window import Window
 
# Define the window specification
window_spec = Window.orderBy(lit(1))
 
# Update the DataFrame with the new Id starting from 294
df = final_df \
    .withColumn("BronzeSchemaName", concat(lit("Concepts_"), col("SourceSchemaName"))) \
    .withColumn("BronzeTableName", col("SourceTableName")) \
    .withColumn("SilverSchemaName", concat(lit("Concepts_"), col("SourceSchemaName"))) \
    .withColumn("SilverTableName", col("SourceTableName")) \
    .withColumn("IsRequired", lit("1")) \
    .withColumn("LoadType", lit("Table")) \
    .withColumn("Id", row_number().over(window_spec) + max_id)  # Adjust Id to start from 294
final_df = df.withColumn("Id", col("Id").cast(IntegerType()))
 
selected_df = final_df.select(
    "Id", "SourceTableName", "SourceSchemaName", "BronzeSchemaName",
    "BronzeTableName", "SilverSchemaName", "SilverTableName",
    "SourceQuery", "IsRequired","LoadType","PrimaryKey"
)
 
selected_df = selected_df.withColumn(
    "CreatedBy",lit(mssparkutils.env.getUserName())
)
createdtime = datetime.now(timezone.utc)
selected_df = selected_df.withColumn(
    "CreatedDate",lit(createdtime).cast(StringType())
)

In [ ]:
################################################### 9. DEFINING STRUCTURE OF THE DATAFRAME ###################################################
 
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
new_schema = StructType([
    StructField("SourceTableName", StringType(), True),
    StructField("SourceSchemaName", StringType(), True),
    StructField("SourceQuery", StringType(), True),
    StructField("BronzeSchemaName", StringType(), True),
    StructField("BronzeTableName", StringType(), True),
    StructField("SilverSchemaName", StringType(), True),
    StructField("SilverTableName", StringType(), True),
    StructField("IsRequired", StringType(), True),
    StructField("LoadType", StringType(), True),
    StructField("Id", IntegerType(), True),
    StructField("PrimaryKey", StringType(), True),
])

df = spark.createDataFrame(df.rdd, new_schema)

In [ ]:
################################################### 9.LOADING THE METADATA DETAILS ###################################################
df.write \
    .mode("append") \
    .synapsesql("WH_MetaData.{ConfigSchemaName}.OneTimeConfigETL")